In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Library

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy
import os

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# 이미지 처리
from skimage import io
import requests
from PIL import Image
from io import BytesIO

# tensorflow 모델
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.optimizers import Adam

# Utils

In [ ]:
# Helper: URL에서 이미지 로드 & (224×224)으로 리사이즈
def load_and_preprocess(url, idx=None, target_size=(224,224)):
    try:
        response = requests.get(url, stream=True, timeout=180)
        img = Image.open(BytesIO(response.content)).resize(target_size)
        img = img.convert("RGB")
        img_array = img_to_array(img)
        img_array = preprocess_input(img_array)
        return img_array
    except Exception as e:
        print(f"[{idx}] Error fetching image: {e}")
        # 에러 인덱스 기록
        if idx is not None:
            error_indices.append(idx)
        img_array = np.zeros((*target_size, 3), dtype=np.float32)
        img_array = preprocess_input(img_array)
        return img_array

In [ ]:
# batch 단위로 feature 추출
def predict_batch(url_list, index_list, batch_size=4, model=None):

    # 1. 이미지 불러오기 및 전처리
    image_arrays = [load_and_preprocess(url, idx) for url, idx in zip(url_list, index_list)]
    batch_array = np.stack(image_arrays, axis=0)

    # 2. 배치 단위로 예측
    preds = model.predict(batch_array, batch_size=batch_size, verbose=1)
    return preds

In [ ]:
# Top 5 core Filtering
def top_n_filtering(data, chk=5):
    cnt = 1
    while True:
        # 사용자 필터링
        user_counts = data['user_id'].value_counts()
        users_to_remove = user_counts[user_counts < chk].index
        data = data[~data['user_id'].isin(users_to_remove)]

        # 아이템 필터링
        item_counts = data['asin'].value_counts()
        items_to_remove = item_counts[item_counts < chk].index
        data = data[~data['asin'].isin(items_to_remove)]

        print(f"Epoch {cnt}")
        print(f" - Remaining users: {data['user_id'].nunique()}")
        print(f" - Remaining items: {data['asin'].nunique()}")
        print(f" - Remaining reviews: {len(data)}")

        # 두 조건 모두 만족 시 종료
        if (len(users_to_remove) == 0) and (len(items_to_remove) == 0):
            break

        cnt += 1

    return data

# Load Dataset

In [ ]:
df = pd.read_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v1.json')
print(df.shape)
df.head()

# Extract VGG16 Image Embedding

In [ ]:
# Load VGG16 model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3), pooling='max')
x = Flatten()(base_model.output)
predictions = Dense(4096, activation='relu')(x)
model = Model(inputs=base_model.input, outputs=predictions)

In [ ]:
# Apply the function to each row
batch_size = 8
all_image_urls = df['medium_image_url'].tolist()
all_indices = df.index.tolist()  # 실제 DataFrame index

error_indices = []
results = []
for i in tqdm(range(0, len(all_image_urls), batch_size)):
    batch_urls = all_image_urls[i:i+batch_size]
    batch_indices = all_indices[i:i+batch_size]
    preds = predict_batch(batch_urls, batch_indices, batch_size=len(batch_urls), model=model)
    results.extend(preds)

embedding = pd.DataFrame({'VGG16_emb': [e for e in results]})
df = pd.concat([df, embedding], axis=1)

In [ ]:
# error_indices에 오류난 데이터 인덱스가 다 들어있음
print(f"\n불러오지 못한 이미지 인덱스: {error_indices}")
print(f"\n총 오류 개수: {len(error_indices)}")
print("\n실패한 url:")

for idx in error_indices:
    print(df.loc[idx, 'medium_image_url'])

In [ ]:
len(error_indices)

In [ ]:
df.shape

In [ ]:
df = df.drop(index=error_indices)
top3 = top_n_filtering(df, 3)

# index 초기화
top3 = top3.reset_index(drop=True)
top3.shape

In [ ]:
top3.to_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v1.json')